# Visual Search System - Serving Architecture

This notebook covers the production serving architecture for a visual search system. We'll explore the prediction pipeline, indexing pipeline, and approximate nearest neighbor algorithms.

## Learning Objectives
- Design the end-to-end serving architecture
- Understand embedding generation and nearest neighbor services
- Learn approximate nearest neighbor (ANN) algorithms
- Implement re-ranking and filtering strategies
- Optimize for latency and memory efficiency

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
from typing import List, Tuple, Dict
from dataclasses import dataclass

## 1. System Architecture Overview

A visual search system has two main pipelines:

1. **Prediction Pipeline (Online)**: Handles real-time search queries
2. **Indexing Pipeline (Offline/Batch)**: Pre-computes embeddings for all images in the catalog

In [ ]:
def visualize_system_architecture():
    """Visualize the visual search system architecture"""
    fig, ax = plt.subplots(1, 1, figsize=(16, 10))
    
    # Colors
    colors = {
        'user': 'lightgreen',
        'online': 'lightblue',
        'storage': 'lightyellow',
        'offline': 'lightcoral'
    }
    
    # Online Pipeline Components
    components = [
        # User
        {'name': 'User\nQuery Image', 'pos': (0.08, 0.5), 'color': colors['user']},
        
        # Online Pipeline
        {'name': 'Preprocessing\nService', 'pos': (0.22, 0.5), 'color': colors['online']},
        {'name': 'Embedding\nService', 'pos': (0.36, 0.5), 'color': colors['online']},
        {'name': 'Nearest\nNeighbor\nService', 'pos': (0.50, 0.5), 'color': colors['online']},
        {'name': 'Re-ranking\nService', 'pos': (0.64, 0.5), 'color': colors['online']},
        {'name': 'Results', 'pos': (0.78, 0.5), 'color': colors['user']},
        
        # Storage
        {'name': 'Vector Index\n(ANN)', 'pos': (0.50, 0.25), 'color': colors['storage']},
        {'name': 'Image\nMetadata', 'pos': (0.64, 0.25), 'color': colors['storage']},
        
        # Offline Pipeline
        {'name': 'Image\nCatalog', 'pos': (0.22, 0.1), 'color': colors['offline']},
        {'name': 'Batch\nEmbedding', 'pos': (0.36, 0.1), 'color': colors['offline']},
        {'name': 'Index\nBuilder', 'pos': (0.50, 0.1), 'color': colors['offline']},
    ]
    
    # Draw components
    for comp in components:
        rect = plt.Rectangle(
            (comp['pos'][0] - 0.06, comp['pos'][1] - 0.08), 0.12, 0.16,
            facecolor=comp['color'], edgecolor='black', linewidth=2
        )
        ax.add_patch(rect)
        ax.text(comp['pos'][0], comp['pos'][1], comp['name'],
                ha='center', va='center', fontsize=9, fontweight='bold')
    
    # Draw arrows (Online Pipeline)
    online_arrows = [
        ((0.14, 0.5), (0.16, 0.5)),   # User to Preprocessing
        ((0.28, 0.5), (0.30, 0.5)),   # Preprocessing to Embedding
        ((0.42, 0.5), (0.44, 0.5)),   # Embedding to NN
        ((0.56, 0.5), (0.58, 0.5)),   # NN to Re-ranking
        ((0.70, 0.5), (0.72, 0.5)),   # Re-ranking to Results
    ]
    
    for start, end in online_arrows:
        ax.annotate('', xy=end, xytext=start,
                   arrowprops=dict(arrowstyle='->', color='blue', lw=2))
    
    # Draw arrows (Storage connections)
    ax.annotate('', xy=(0.50, 0.33), xytext=(0.50, 0.42),
               arrowprops=dict(arrowstyle='<->', color='green', lw=2))
    ax.annotate('', xy=(0.64, 0.33), xytext=(0.64, 0.42),
               arrowprops=dict(arrowstyle='<->', color='green', lw=2))
    
    # Draw arrows (Offline Pipeline)
    offline_arrows = [
        ((0.28, 0.1), (0.30, 0.1)),   # Catalog to Batch
        ((0.42, 0.1), (0.44, 0.1)),   # Batch to Index Builder
        ((0.50, 0.18), (0.50, 0.17)), # Index Builder to Vector Index
    ]
    
    for start, end in offline_arrows:
        ax.annotate('', xy=end, xytext=start,
                   arrowprops=dict(arrowstyle='->', color='red', lw=2))
    
    # Labels
    ax.text(0.5, 0.72, 'ONLINE PREDICTION PIPELINE', ha='center', fontsize=14, 
            fontweight='bold', color='blue')
    ax.text(0.36, 0.02, 'OFFLINE INDEXING PIPELINE', ha='center', fontsize=14, 
            fontweight='bold', color='red')
    
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 0.8)
    ax.axis('off')
    ax.set_title('Visual Search System Architecture', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

visualize_system_architecture()

## 2. Prediction Pipeline (Online)

The prediction pipeline handles real-time search queries with strict latency requirements.

### 2.1 Preprocessing Service
- Resize image to model input size (e.g., 224×224)
- Normalize pixel values
- Handle image cropping if user selects a region

In [ ]:
@dataclass
class ImageRequest:
    image_bytes: bytes
    crop_box: Tuple[int, int, int, int] = None  # (x1, y1, x2, y2)

class PreprocessingService:
    """
    Handles image preprocessing for the embedding model.
    """
    def __init__(self, target_size=(224, 224), normalize=True):
        self.target_size = target_size
        self.normalize = normalize
        self.mean = np.array([0.485, 0.456, 0.406])
        self.std = np.array([0.229, 0.224, 0.225])
    
    def preprocess(self, image: np.ndarray, crop_box: Tuple = None) -> np.ndarray:
        """
        Preprocess image for embedding model.
        
        Steps:
        1. Apply crop if specified
        2. Resize to target size
        3. Convert to RGB if needed
        4. Normalize pixel values
        """
        # Simulate preprocessing
        processed = {
            'original_shape': image.shape,
            'crop_applied': crop_box is not None,
            'target_size': self.target_size,
            'normalized': self.normalize
        }
        return processed

# Example
preprocessing = PreprocessingService()
sample_image = np.random.randint(0, 255, (800, 600, 3), dtype=np.uint8)
result = preprocessing.preprocess(sample_image, crop_box=(100, 100, 400, 400))
print("Preprocessing Steps:")
for key, value in result.items():
    print(f"  {key}: {value}")

### 2.2 Embedding Service

Runs the neural network model to generate embeddings.

In [ ]:
class EmbeddingService:
    """
    Generates image embeddings using the trained model.
    """
    def __init__(self, model_path: str, embedding_dim: int = 128):
        self.model_path = model_path
        self.embedding_dim = embedding_dim
        # In production, would load actual model here
        self.model = None
    
    def generate_embedding(self, preprocessed_image: np.ndarray) -> np.ndarray:
        """
        Generate embedding vector from preprocessed image.
        
        Returns:
            Normalized embedding vector of shape (embedding_dim,)
        """
        # Simulate embedding generation
        start_time = time.time()
        
        # In production: embedding = self.model.predict(preprocessed_image)
        embedding = np.random.randn(self.embedding_dim)
        
        # L2 normalize
        embedding = embedding / np.linalg.norm(embedding)
        
        latency = (time.time() - start_time) * 1000
        return embedding, latency

# Example
embedding_service = EmbeddingService("model.pt", embedding_dim=128)
embedding, latency = embedding_service.generate_embedding(np.zeros((224, 224, 3)))
print(f"Embedding shape: {embedding.shape}")
print(f"Embedding norm: {np.linalg.norm(embedding):.4f} (should be 1.0)")
print(f"Inference latency: {latency:.2f}ms")

### 2.3 Nearest Neighbor Service

Finds the most similar images from the index.

In [ ]:
class NearestNeighborService:
    """
    Retrieves similar images using ANN search.
    """
    def __init__(self, index, k: int = 100):
        self.index = index
        self.default_k = k
    
    def search(self, query_embedding: np.ndarray, k: int = None) -> List[Tuple[str, float]]:
        """
        Find k nearest neighbors.
        
        Returns:
            List of (image_id, similarity_score) tuples
        """
        if k is None:
            k = self.default_k
        
        # In production: results = self.index.search(query_embedding, k)
        # Simulate results
        results = [
            (f"img_{i:06d}", 1.0 - i * 0.01)
            for i in range(k)
        ]
        return results

# Example
nn_service = NearestNeighborService(None, k=10)
results = nn_service.search(np.random.randn(128))
print("Top 10 Nearest Neighbors:")
for img_id, score in results[:10]:
    print(f"  {img_id}: similarity = {score:.4f}")

### 2.4 Re-ranking Service

Applies business logic, filtering, and re-ranking to the initial results.

In [ ]:
class ReRankingService:
    """
    Re-ranks and filters search results based on business rules.
    """
    def __init__(self, filters: List[str] = None):
        self.filters = filters or []
    
    def rerank(self, results: List[Tuple[str, float]], 
               user_context: Dict = None) -> List[Tuple[str, float]]:
        """
        Apply re-ranking logic:
        1. Remove duplicates
        2. Filter inappropriate content
        3. Apply diversity (avoid too similar results)
        4. Boost/demote based on business rules
        """
        reranked = []
        seen_hashes = set()
        
        for img_id, score in results:
            # Simulate deduplication
            if img_id in seen_hashes:
                continue
            seen_hashes.add(img_id)
            
            # Simulate content filtering (skip 10% randomly)
            if np.random.random() < 0.1:
                continue
            
            reranked.append((img_id, score))
        
        return reranked

# Example
rerank_service = ReRankingService()
initial_results = [(f"img_{i}", 0.9 - i*0.05) for i in range(20)]
final_results = rerank_service.rerank(initial_results)

print(f"Initial results: {len(initial_results)}")
print(f"After re-ranking: {len(final_results)}")
print("\nFinal top 5:")
for img_id, score in final_results[:5]:
    print(f"  {img_id}: {score:.4f}")

## 3. Indexing Pipeline (Offline)

The indexing pipeline pre-computes embeddings for all images and builds the search index.

In [ ]:
class IndexingPipeline:
    """
    Batch processing pipeline for building the search index.
    """
    def __init__(self, embedding_service: EmbeddingService):
        self.embedding_service = embedding_service
    
    def process_catalog(self, image_paths: List[str], batch_size: int = 256):
        """
        Process all images in the catalog:
        1. Load images in batches
        2. Generate embeddings
        3. Store embeddings with image IDs
        """
        all_embeddings = []
        all_ids = []
        
        n_batches = (len(image_paths) + batch_size - 1) // batch_size
        
        for batch_idx in range(n_batches):
            start = batch_idx * batch_size
            end = min(start + batch_size, len(image_paths))
            batch_paths = image_paths[start:end]
            
            # Process batch
            for path in batch_paths:
                # Simulate: load image, preprocess, generate embedding
                embedding = np.random.randn(self.embedding_service.embedding_dim)
                embedding = embedding / np.linalg.norm(embedding)
                
                all_embeddings.append(embedding)
                all_ids.append(path)
        
        return np.array(all_embeddings), all_ids
    
    def build_index(self, embeddings: np.ndarray, ids: List[str]):
        """
        Build ANN index from embeddings.
        """
        index_info = {
            'num_vectors': len(ids),
            'embedding_dim': embeddings.shape[1],
            'index_type': 'HNSW',  # or IVF, etc.
            'memory_mb': (embeddings.nbytes / 1024 / 1024)
        }
        return index_info

# Example
indexing = IndexingPipeline(EmbeddingService("model.pt"))

# Simulate 10000 images
sample_paths = [f"images/img_{i:06d}.jpg" for i in range(10000)]
embeddings, ids = indexing.process_catalog(sample_paths, batch_size=1000)

print(f"Processed {len(ids)} images")
print(f"Embeddings shape: {embeddings.shape}")

index_info = indexing.build_index(embeddings, ids)
print("\nIndex Info:")
for key, value in index_info.items():
    print(f"  {key}: {value}")

## 4. Approximate Nearest Neighbor (ANN) Algorithms

For large-scale visual search (billions of images), exact nearest neighbor search is too slow. We use ANN algorithms that trade some accuracy for much faster search.

### 4.1 Exact vs Approximate Search

| Method | Time Complexity | Recall | Use Case |
|--------|-----------------|--------|----------|
| Exact (Linear) | O(N × D) | 100% | Small datasets (<100K) |
| Approximate | O(log N) to O(√N) | 90-99% | Large datasets |

In [ ]:
def compare_search_complexity():
    """Visualize search complexity comparison"""
    n_values = np.logspace(3, 10, 100)  # 1K to 10B
    
    # Time complexity (relative)
    exact_time = n_values  # O(N)
    hnsw_time = np.log(n_values)  # O(log N)
    ivf_time = np.sqrt(n_values)  # O(sqrt N)
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    ax.loglog(n_values, exact_time, 'r-', linewidth=2, label='Exact Search O(N)')
    ax.loglog(n_values, hnsw_time * 1e4, 'b-', linewidth=2, label='HNSW O(log N)')
    ax.loglog(n_values, ivf_time * 1e2, 'g-', linewidth=2, label='IVF O(√N)')
    
    # Mark key points
    ax.axvline(x=1e6, color='gray', linestyle='--', alpha=0.5)
    ax.axvline(x=1e9, color='gray', linestyle='--', alpha=0.5)
    ax.text(1e6, 1e8, '1M images', rotation=90, va='bottom', ha='right')
    ax.text(1e9, 1e8, '1B images', rotation=90, va='bottom', ha='right')
    
    ax.set_xlabel('Number of Images', fontsize=12)
    ax.set_ylabel('Relative Search Time', fontsize=12)
    ax.set_title('Search Time Complexity Comparison', fontsize=14)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

compare_search_complexity()

### 4.2 ANN Algorithm Categories

#### Tree-based Methods
- **KD-Trees**: Partition space recursively. Good for low dimensions.
- **Annoy (Spotify)**: Multiple random projection trees. Good balance of speed/accuracy.

#### Hash-based Methods
- **Locality-Sensitive Hashing (LSH)**: Hash similar items to same buckets.

#### Graph-based Methods
- **HNSW (Hierarchical Navigable Small World)**: Build a navigable graph. State-of-the-art accuracy.

#### Quantization-based Methods
- **IVF (Inverted File Index)**: Cluster embeddings, search within relevant clusters.
- **PQ (Product Quantization)**: Compress vectors for memory efficiency.

In [ ]:
def visualize_ann_algorithms():
    """Visualize different ANN approaches"""
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    
    np.random.seed(42)
    points = np.random.randn(100, 2)
    query = np.array([0, 0])
    
    # 1. KD-Tree (space partitioning)
    ax1 = axes[0, 0]
    ax1.scatter(points[:, 0], points[:, 1], c='blue', alpha=0.6, s=30)
    ax1.scatter(*query, c='red', s=100, marker='*', label='Query')
    ax1.axhline(y=0, color='green', linewidth=2)
    ax1.axvline(x=-1, color='green', linewidth=2, linestyle='--')
    ax1.axvline(x=1, color='green', linewidth=2, linestyle='--')
    ax1.set_title('KD-Tree: Space Partitioning', fontsize=12)
    ax1.set_xlim(-3, 3)
    ax1.set_ylim(-3, 3)
    ax1.legend()
    
    # 2. LSH (hash buckets)
    ax2 = axes[0, 1]
    ax2.scatter(points[:, 0], points[:, 1], c='blue', alpha=0.6, s=30)
    ax2.scatter(*query, c='red', s=100, marker='*', label='Query')
    # Draw hash buckets
    for i in range(-3, 4):
        ax2.axhline(y=i, color='green', alpha=0.3, linewidth=1)
        ax2.axvline(x=i, color='green', alpha=0.3, linewidth=1)
    # Highlight query bucket
    rect = plt.Rectangle((-1, -1), 2, 2, fill=True, facecolor='yellow', alpha=0.3)
    ax2.add_patch(rect)
    ax2.set_title('LSH: Hash Buckets', fontsize=12)
    ax2.set_xlim(-3, 3)
    ax2.set_ylim(-3, 3)
    ax2.legend()
    
    # 3. HNSW (graph navigation)
    ax3 = axes[1, 0]
    ax3.scatter(points[:, 0], points[:, 1], c='blue', alpha=0.6, s=30)
    ax3.scatter(*query, c='red', s=100, marker='*', label='Query')
    # Draw graph connections (simplified)
    for i in range(20):
        for j in range(i+1, min(i+4, 20)):
            ax3.plot([points[i, 0], points[j, 0]], 
                    [points[i, 1], points[j, 1]], 
                    'g-', alpha=0.2, linewidth=0.5)
    # Highlight search path
    path_idx = [50, 30, 10, 5]
    for i in range(len(path_idx)-1):
        ax3.plot([points[path_idx[i], 0], points[path_idx[i+1], 0]], 
                [points[path_idx[i], 1], points[path_idx[i+1], 1]], 
                'r-', linewidth=2)
    ax3.set_title('HNSW: Graph Navigation', fontsize=12)
    ax3.set_xlim(-3, 3)
    ax3.set_ylim(-3, 3)
    ax3.legend()
    
    # 4. IVF (clustering)
    ax4 = axes[1, 1]
    # Assign clusters
    from scipy.cluster.hierarchy import fclusterdata
    clusters = fclusterdata(points, t=4, criterion='maxclust')
    ax4.scatter(points[:, 0], points[:, 1], c=clusters, cmap='viridis', alpha=0.6, s=30)
    ax4.scatter(*query, c='red', s=100, marker='*', label='Query')
    # Draw cluster centers
    for c in range(1, 5):
        mask = clusters == c
        center = points[mask].mean(axis=0)
        ax4.scatter(*center, c='black', s=150, marker='x', linewidths=3)
    ax4.set_title('IVF: Cluster-based Search', fontsize=12)
    ax4.set_xlim(-3, 3)
    ax4.set_ylim(-3, 3)
    ax4.legend()
    
    plt.suptitle('Approximate Nearest Neighbor Algorithms', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

visualize_ann_algorithms()

### 4.3 Popular ANN Libraries

| Library | Developed By | Best For | Key Features |
|---------|-------------|----------|---------------|
| **Faiss** | Meta | Large-scale, GPU | IVF, PQ, HNSW |
| **ScaNN** | Google | High recall | Learned quantization |
| **Annoy** | Spotify | Memory-mapped | Random projection trees |
| **Milvus** | Zilliz | Distributed | Vector database |
| **Pinecone** | Pinecone | Managed service | Serverless |

In [ ]:
# Example: Faiss-like index configuration (pseudocode)
def faiss_index_configurations():
    """
    Common Faiss index configurations for different scales.
    """
    configs = [
        {
            'scale': 'Small (<1M vectors)',
            'index_type': 'IndexFlatL2',
            'description': 'Exact search, brute force',
            'recall': '100%',
            'memory': 'High (4 bytes × dim × N)',
            'speed': 'Slow'
        },
        {
            'scale': 'Medium (1-100M vectors)',
            'index_type': 'IndexIVFFlat',
            'description': 'Inverted file with flat quantizer',
            'recall': '95-99%',
            'memory': 'High',
            'speed': 'Fast'
        },
        {
            'scale': 'Large (100M-1B vectors)',
            'index_type': 'IndexIVFPQ',
            'description': 'IVF + Product Quantization',
            'recall': '90-95%',
            'memory': 'Low (compressed)',
            'speed': 'Very Fast'
        },
        {
            'scale': 'Any (best quality)',
            'index_type': 'IndexHNSWFlat',
            'description': 'Hierarchical NSW graph',
            'recall': '95-99%',
            'memory': 'High + graph overhead',
            'speed': 'Very Fast'
        }
    ]
    
    print("Faiss Index Configurations:")
    print("=" * 70)
    
    for config in configs:
        print(f"\n{config['scale']}")
        print(f"  Index Type: {config['index_type']}")
        print(f"  Description: {config['description']}")
        print(f"  Recall: {config['recall']}")
        print(f"  Memory: {config['memory']}")
        print(f"  Speed: {config['speed']}")

faiss_index_configurations()

## 5. Memory Optimization

At scale (100B images), storing full embeddings is expensive. We use quantization to reduce memory.

### 5.1 Vector Quantization
Reduce float32 (4 bytes) to int8 (1 byte) or less.

In [ ]:
def calculate_memory_requirements():
    """
    Calculate memory requirements for different configurations.
    """
    num_images = 100_000_000_000  # 100B images
    embedding_dim = 128
    
    configurations = [
        ('float32 (full precision)', 4),
        ('float16 (half precision)', 2),
        ('int8 (quantized)', 1),
        ('PQ (64 bytes)', 64 / embedding_dim),
        ('PQ (32 bytes)', 32 / embedding_dim),
    ]
    
    print(f"Memory Requirements for {num_images/1e9:.0f}B images × {embedding_dim} dims")
    print("=" * 60)
    
    results = []
    for name, bytes_per_dim in configurations:
        memory_bytes = num_images * embedding_dim * bytes_per_dim
        memory_tb = memory_bytes / (1024**4)
        results.append((name, memory_tb))
        print(f"{name:30s}: {memory_tb:>10.1f} TB")
    
    # Visualize
    fig, ax = plt.subplots(figsize=(10, 5))
    names, memory = zip(*results)
    bars = ax.barh(names, memory, color='steelblue', edgecolor='black')
    ax.set_xlabel('Memory (TB)', fontsize=12)
    ax.set_title('Memory Requirements: 100B Images × 128 dims', fontsize=14)
    
    for bar, mem in zip(bars, memory):
        ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
               f'{mem:.1f} TB', va='center', fontsize=10)
    
    ax.set_xlim(0, max(memory) * 1.2)
    plt.tight_layout()
    plt.show()

calculate_memory_requirements()

### 5.2 Product Quantization (PQ)

Split embedding into sub-vectors and quantize each separately.

In [ ]:
def explain_product_quantization():
    """Explain Product Quantization concept"""
    explanation = """
    PRODUCT QUANTIZATION (PQ)
    ═════════════════════════
    
    Original embedding: [v1, v2, v3, v4, ..., v128] (128 dims × 4 bytes = 512 bytes)
    
    Step 1: Split into M sub-vectors (e.g., M=8)
    ─────────────────────────────────────────────
    Sub-vector 1: [v1, ..., v16]
    Sub-vector 2: [v17, ..., v32]
    ...
    Sub-vector 8: [v113, ..., v128]
    
    Step 2: Learn K centroids for each sub-space (e.g., K=256)
    ─────────────────────────────────────────────────────────
    Codebook 1: 256 centroids in 16-dim space
    Codebook 2: 256 centroids in 16-dim space
    ...
    Codebook 8: 256 centroids in 16-dim space
    
    Step 3: Encode each sub-vector as nearest centroid index
    ─────────────────────────────────────────────────────────
    Original: [v1...v16] → Nearest centroid index: 42 (1 byte)
    
    Final encoding: 8 bytes total (8 sub-vectors × 1 byte each)
    
    COMPRESSION RATIO: 512 bytes → 8 bytes = 64× compression!
    
    SEARCH:
    1. Pre-compute distances from query to all centroids
    2. Approximate distance = sum of sub-vector distances
    3. Much faster than computing full distance!
    """
    print(explanation)

explain_product_quantization()

## 6. Latency Optimization

### 6.1 Latency Budget

In [ ]:
def latency_breakdown():
    """Typical latency breakdown for visual search"""
    components = [
        ('Network (upload image)', 50),
        ('Preprocessing', 5),
        ('Embedding generation (GPU)', 20),
        ('ANN search', 10),
        ('Re-ranking', 5),
        ('Metadata lookup', 5),
        ('Network (return results)', 5),
    ]
    
    names, latencies = zip(*components)
    total = sum(latencies)
    
    print("Visual Search Latency Breakdown")
    print("=" * 50)
    
    for name, latency in components:
        pct = (latency / total) * 100
        print(f"{name:35s}: {latency:>3d}ms ({pct:>5.1f}%)")
    
    print("=" * 50)
    print(f"{'TOTAL':35s}: {total:>3d}ms")
    
    # Visualize
    fig, ax = plt.subplots(figsize=(10, 6))
    colors = plt.cm.viridis(np.linspace(0, 0.8, len(components)))
    
    bottom = 0
    for (name, latency), color in zip(components, colors):
        ax.barh(['Latency'], [latency], left=bottom, color=color, edgecolor='black', 
               label=f"{name} ({latency}ms)")
        bottom += latency
    
    ax.set_xlabel('Latency (ms)', fontsize=12)
    ax.set_title('Visual Search End-to-End Latency', fontsize=14)
    ax.legend(loc='center left', bbox_to_anchor=(1, 0.5), fontsize=9)
    ax.axvline(x=100, color='red', linestyle='--', linewidth=2, label='Target: 100ms')
    
    plt.tight_layout()
    plt.show()

latency_breakdown()

### 6.2 Optimization Techniques

In [ ]:
def optimization_techniques():
    """
    List of optimization techniques for visual search serving.
    """
    techniques = """
    SERVING OPTIMIZATION TECHNIQUES
    ════════════════════════════════
    
    1. MODEL OPTIMIZATION
       ├── Model distillation (smaller model)
       ├── Quantization (FP16, INT8)
       ├── TensorRT/ONNX optimization
       └── Batch inference on GPU
    
    2. INDEX OPTIMIZATION
       ├── Use appropriate ANN algorithm
       ├── Tune nprobe/efSearch parameters
       ├── Shard index across machines
       └── Keep hot data in memory
    
    3. CACHING
       ├── Cache popular query embeddings
       ├── Cache search results
       └── CDN for image thumbnails
    
    4. INFRASTRUCTURE
       ├── GPU inference servers
       ├── Load balancing
       ├── Geographic distribution
       └── Auto-scaling
    
    5. PREPROCESSING
       ├── Client-side image resizing
       ├── Progressive image loading
       └── Async preprocessing pipeline
    """
    print(techniques)

optimization_techniques()

## 7. Complete Prediction Pipeline Code

In [ ]:
class VisualSearchPipeline:
    """
    Complete visual search prediction pipeline.
    """
    def __init__(self, config: Dict):
        self.preprocessing = PreprocessingService(
            target_size=config.get('image_size', (224, 224))
        )
        self.embedding = EmbeddingService(
            model_path=config.get('model_path', 'model.pt'),
            embedding_dim=config.get('embedding_dim', 128)
        )
        self.nn_service = NearestNeighborService(
            index=None,  # Would load actual index
            k=config.get('k', 100)
        )
        self.reranker = ReRankingService()
    
    def search(self, image: np.ndarray, crop_box: Tuple = None, 
               k: int = 10) -> Dict:
        """
        Execute visual search pipeline.
        
        Returns:
            Dict with results and timing info
        """
        timings = {}
        
        # Step 1: Preprocessing
        start = time.time()
        preprocessed = self.preprocessing.preprocess(image, crop_box)
        timings['preprocessing_ms'] = (time.time() - start) * 1000
        
        # Step 2: Generate embedding
        start = time.time()
        embedding, _ = self.embedding.generate_embedding(preprocessed)
        timings['embedding_ms'] = (time.time() - start) * 1000
        
        # Step 3: ANN search
        start = time.time()
        candidates = self.nn_service.search(embedding, k=k*2)
        timings['search_ms'] = (time.time() - start) * 1000
        
        # Step 4: Re-ranking
        start = time.time()
        results = self.reranker.rerank(candidates)
        timings['reranking_ms'] = (time.time() - start) * 1000
        
        timings['total_ms'] = sum(timings.values())
        
        return {
            'results': results[:k],
            'timings': timings,
            'num_candidates': len(candidates),
            'num_results': len(results[:k])
        }

# Example usage
config = {
    'image_size': (224, 224),
    'model_path': 'resnet50_embedding.pt',
    'embedding_dim': 128,
    'k': 100
}

pipeline = VisualSearchPipeline(config)

# Simulate search
sample_image = np.random.randint(0, 255, (640, 480, 3), dtype=np.uint8)
response = pipeline.search(sample_image, k=10)

print("Visual Search Response:")
print("=" * 50)
print(f"\nCandidates retrieved: {response['num_candidates']}")
print(f"Final results: {response['num_results']}")
print("\nTimings:")
for stage, ms in response['timings'].items():
    print(f"  {stage}: {ms:.2f}ms")
print("\nTop 5 Results:")
for img_id, score in response['results'][:5]:
    print(f"  {img_id}: similarity = {score:.4f}")

## 8. Summary

### Key Takeaways

1. **Architecture Components:**
   - Preprocessing → Embedding → ANN Search → Re-ranking
   - Separate online (prediction) and offline (indexing) pipelines

2. **ANN Algorithms:**
   - HNSW for best accuracy
   - IVF+PQ for memory efficiency
   - Use Faiss or ScaNN for production

3. **Memory Optimization:**
   - Product Quantization can achieve 64× compression
   - Trade accuracy for memory when needed

4. **Latency Optimization:**
   - Target <100ms end-to-end
   - GPU for embedding generation
   - Caching for popular queries

In [ ]:
def serving_architecture_summary():
    summary = """
    ╔══════════════════════════════════════════════════════════════╗
    ║       VISUAL SEARCH SERVING ARCHITECTURE SUMMARY             ║
    ╠══════════════════════════════════════════════════════════════╣
    ║                                                              ║
    ║   ONLINE PIPELINE:                                           ║
    ║   User → Preprocess → Embed → ANN Search → Re-rank → Results║
    ║                                                              ║
    ║   OFFLINE PIPELINE:                                          ║
    ║   Catalog → Batch Embed → Build Index → Deploy               ║
    ║                                                              ║
    ║   ANN ALGORITHMS:                                            ║
    ║   ├── HNSW: Best recall, graph-based                        ║
    ║   ├── IVF: Fast, cluster-based                              ║
    ║   └── PQ: Memory efficient, 64× compression                 ║
    ║                                                              ║
    ║   OPTIMIZATION:                                              ║
    ║   ├── GPU inference for embeddings                          ║
    ║   ├── Quantization for memory                               ║
    ║   └── Caching for latency                                   ║
    ║                                                              ║
    ║   SCALE: 100B images, <100ms latency                        ║
    ║                                                              ║
    ╚══════════════════════════════════════════════════════════════╝
    """
    print(summary)

serving_architecture_summary()